# Regresi: Memprediksi Harga Rumah

**Buku:** *Deep Learning dengan Python* — Bab 3, Memulai dengan Jaringan Saraf

Memprediksi nilai kontinu (median harga rumah) dari 13 fitur numerik, dengan normalisasi fitur dan validasi *K-fold*.

> Diadaptasi dari notebook pendamping *Deep Learning with Python* (F. Chollet, Manning; kode berlisensi MIT) dan diperbarui ke **Keras 3** untuk buku *Deep Learning dengan Python: Konsep, Kode, dan Penerapan Modern* (B. S. Negara).

**Lingkungan:** Google Colab (GPU disarankan), Python 3.10+, Keras 3. Kode bersifat *backend-agnostik*: dapat dijalankan dengan backend TensorFlow, JAX, atau PyTorch.

**Perubahan utama dari versi asli (Keras 2.0.8):**
- Kunci riwayat `val_mean_absolute_error` → `val_mae` (asli akan *KeyError* di Keras modern).
- `K.clear_session()` → `keras.backend.clear_session()`.
- Normalisasi tidak lagi mengubah larik asli secara *in-place* (aman dijalankan ulang).
- Ukuran batch K-fold 1 → 16 agar jauh lebih cepat; epoch terbaik dipilih otomatis dari kurva.
- Catatan etika: dataset Boston memuat fitur berbasis ras dan sudah dihapus dari scikit-learn; dipakai di sini hanya sebagai contoh historis.

In [ ]:
import os
# Pilih backend: "tensorflow", "jax", atau "torch"
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
print("Keras", keras.__version__, "| backend:", keras.backend.backend())

In [ ]:
import numpy as np
from keras.datasets import boston_housing

(data_latih, target_latih), (data_uji, target_uji) = boston_housing.load_data()
print(data_latih.shape, data_uji.shape)
print("Contoh target (ribu USD):", target_latih[:5])

## Normalisasi per fitur
Rerata dan simpangan baku dihitung **hanya dari data latih**.

In [ ]:
rerata = data_latih.mean(axis=0)
simpangan = data_latih.std(axis=0)
x_latih = (data_latih - rerata) / simpangan
x_uji = (data_uji - rerata) / simpangan

In [ ]:
from keras import layers

def bangun_model():
    model = keras.Sequential([
        keras.Input(shape=(x_latih.shape[1],)),
        layers.Dense(64, activation="relu"),
        layers.Dense(64, activation="relu"),
        layers.Dense(1),                     # tanpa aktivasi: regresi skalar
    ])
    model.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])
    return model

## Validasi K-fold (K = 4)

In [ ]:
k = 4
n_val = len(x_latih) // k
EPOCH = 500
riwayat_mae = []
for i in range(k):
    print("Fold", i)
    x_val = x_latih[i * n_val:(i + 1) * n_val]
    y_val = target_latih[i * n_val:(i + 1) * n_val]
    x_sisa = np.concatenate([x_latih[:i * n_val], x_latih[(i + 1) * n_val:]])
    y_sisa = np.concatenate([target_latih[:i * n_val], target_latih[(i + 1) * n_val:]])
    keras.backend.clear_session()
    model = bangun_model()
    r = model.fit(x_sisa, y_sisa, validation_data=(x_val, y_val),
                  epochs=EPOCH, batch_size=16, verbose=0)
    riwayat_mae.append(r.history["val_mae"])

rata_mae = np.mean(riwayat_mae, axis=0)
print(f"MAE validasi terbaik: {rata_mae.min():.2f} pada epoch {rata_mae.argmin() + 1}")

In [ ]:
import matplotlib.pyplot as plt

def haluskan(titik, faktor=0.9):
    hasil = []
    for t in titik:
        hasil.append(hasil[-1] * faktor + t * (1 - faktor) if hasil else t)
    return hasil

MULAI = min(10, len(rata_mae) // 2)   # buang epoch awal yang nilainya jauh lebih besar
plt.plot(range(MULAI + 1, len(rata_mae) + 1), haluskan(rata_mae[MULAI:]))
plt.xlabel("Epoch"); plt.ylabel("MAE validasi (dihaluskan)"); plt.show()

## Model akhir
Latih pada seluruh data latih dengan jumlah epoch terbaik, lalu uji **sekali**.

In [ ]:
epoch_terbaik = int(np.argmin(haluskan(rata_mae[MULAI:]))) + MULAI + 1
model = bangun_model()
model.fit(x_latih, target_latih, epochs=epoch_terbaik, batch_size=16, verbose=0)
mse_uji, mae_uji = model.evaluate(x_uji, target_uji, verbose=0)
print(f"Epoch: {epoch_terbaik} | MAE uji: {mae_uji:.2f} (≈ USD {mae_uji * 1000:,.0f})")